# Hello MinIO + Spark

This notebook reads a CSV from MinIO through S3A, aggregates it with PySpark, and writes the result back to MinIO.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum as sum_

spark = (
    SparkSession.builder
    .appName('hello-minio-spark')
    .master('spark://spark-master:7077')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

In [ ]:
input_df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv('s3a://datasets/sample.csv')
)
input_df.show()

In [ ]:
result_df = input_df.groupBy('category').agg(
    count('*').alias('row_count'),
    sum_('amount').alias('total_amount'),
)
result_df.show()

In [ ]:
result_df.write.mode('overwrite').parquet('s3a://student-results/hello-minio-spark/')
print('Result written to MinIO.')